# Partial Post-Selection Sweep for `N20x40`

This Colab/A100 notebook runs canonical GPU Markov dynamics through `classA_U1FGTN_gpu.run_markov_circuit(...)` and computes streamed observables for a partial post-selection probability sweep.

Geometry and protocol:
- `Nx=20`, `Ny=40`, `cycles=Ny=40`
- `nshell=1`, `dw_truncation=True`, `alpha_2=30`
- `postselect_probability` in `{0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1}`
- `samples=10` for `p<1`; deterministic post-selection `p=1` runs with one actual sample.

Saved per-run products:
- `scalar_metrics.csv`
- `entropy_y0avg_vs_ay.npz`
- `xavg_square_correlator_vs_ry.npz`
- `local_charge_cell.npz`
- `born_rule_probabilities.npz` where enabled
- `measurement_frustration.npz` for `p<1`, including `postselect_forced_site`
- `run_summary.json`
- `latest_run.json`


In [ ]:
from __future__ import annotations

import json
from pathlib import Path
import sys

try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)

BUNDLE_NAME = 'colab_partial_post-select'


def is_bundle_root(path: Path) -> bool:
    return (
        (path / 'src' / 'classA_U1FGTN_gpu.py').exists()
        and (path / 'src' / 'streaming_covariance_observables_gpu.py').exists()
    )


candidates = []
for base in [Path.cwd(), *Path.cwd().parents]:
    candidates.extend([base, base / BUNDLE_NAME])
if IN_COLAB:
    candidates.extend([
        Path('/content/drive/MyDrive') / BUNDLE_NAME,
        Path('/content/drive/MyDrive/class_A_fermionic_adaptive_circuit') / BUNDLE_NAME,
    ])

seen = set()
BUNDLE_ROOT = None
for candidate in candidates:
    candidate = candidate.resolve()
    if candidate in seen:
        continue
    seen.add(candidate)
    if is_bundle_root(candidate):
        BUNDLE_ROOT = candidate
        break
if BUNDLE_ROOT is None:
    raise FileNotFoundError(f'Could not locate {BUNDLE_NAME} with required src files')

PROJECT_ROOT = BUNDLE_ROOT.parent
BUNDLE_SRC_DIR = BUNDLE_ROOT / 'src'
for path in [BUNDLE_SRC_DIR]:
    if path.exists() and str(path) not in sys.path:
        sys.path.insert(0, str(path))

print(f'[setup] bundle root: {BUNDLE_ROOT}')
print(f'[setup] project root: {PROJECT_ROOT}')


In [ ]:
import inspect
import os
import shutil
import time
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from tqdm.auto import tqdm
import torch

from classA_U1FGTN_gpu import classA_U1FGTN_gpu
import streaming_covariance_observables_gpu as streaming_covariance_helper
from streaming_covariance_observables_gpu import (
    HELPER_VERSION,
    StreamingCovarianceObservables,
    domain_wall_metadata,
    expected_samples_for_config,
    geometry_key,
    rel_to_root,
    run_internal_checks,
    save_dataframe_atomic_csv,
    save_npz_atomic,
    write_json_atomic,
)

EXPECTED_HELPER_VERSION = 'streaming_covariance_observables_gpu_v4'
EXPECTED_HELPER_SOURCE = (BUNDLE_SRC_DIR / 'streaming_covariance_observables_gpu.py').resolve()
HELPER_MODULE_PATH = Path(inspect.getsourcefile(streaming_covariance_helper) or streaming_covariance_helper.__file__).resolve()
HELPER_SIGNATURE = inspect.signature(StreamingCovarianceObservables.make_cycle_observer)
HELPER_INIT_SIGNATURE = inspect.signature(StreamingCovarianceObservables.__init__)

if HELPER_VERSION != EXPECTED_HELPER_VERSION:
    raise RuntimeError(
        'Loaded the wrong streaming covariance helper version: '
        f'expected {EXPECTED_HELPER_VERSION}, got {HELPER_VERSION}. '
        'Re-upload colab_partial_post-select/src/streaming_covariance_observables_gpu.py '
        'and restart the Colab runtime.'
    )
if HELPER_MODULE_PATH != EXPECTED_HELPER_SOURCE:
    raise RuntimeError(
        'Loaded streaming_covariance_observables_gpu from the wrong path: '
        f'{HELPER_MODULE_PATH}. Expected {EXPECTED_HELPER_SOURCE}. '
        'Re-upload colab_partial_post-select/src/streaming_covariance_observables_gpu.py '
        'and restart the Colab runtime.'
    )
if 'model' not in HELPER_SIGNATURE.parameters:
    raise RuntimeError(
        'Loaded a stale StreamingCovarianceObservables.make_cycle_observer signature '
        f'{HELPER_SIGNATURE}. Re-upload '
        'colab_partial_post-select/src/streaming_covariance_observables_gpu.py '
        'and restart the Colab runtime.'
    )
if 'enable_born_probabilities' not in HELPER_INIT_SIGNATURE.parameters:
    raise RuntimeError(
        'Loaded a stale StreamingCovarianceObservables.__init__ signature '
        f'{HELPER_INIT_SIGNATURE}. Re-upload '
        'colab_partial_post-select/src/streaming_covariance_observables_gpu.py '
        'and restart the Colab runtime.'
    )

if not torch.cuda.is_available():
    raise RuntimeError('CUDA is required for this Colab GPU campaign.')

torch.set_grad_enabled(False)
plt.rcParams.update({
    'figure.dpi': 130,
    'savefig.dpi': 180,
    'axes.grid': True,
    'grid.alpha': 0.25,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

total_gib = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
print(f'[setup] helper={HELPER_VERSION}')
print(f'[setup] helper source: {HELPER_MODULE_PATH}')
print(f'[setup] helper signature: {HELPER_SIGNATURE}')
print(f'[setup] helper init signature: {HELPER_INIT_SIGNATURE}')
print(f'[setup] torch={torch.__version__}')
print(f'[setup] cuda device: {torch.cuda.get_device_name(0)}')
print(f'[setup] total gpu memory: {total_gib:.2f} GiB')
if total_gib < 35.0:
    print('[warn] This notebook is sized for an A100 40GB runtime.')

run_internal_checks(include_local_chern_marker=False)
print('[setup] helper internal checks passed')


## Campaign Configuration

The p-sweep treats `p<1` as trajectory-resolved perfect-correction dynamics with a per-site forced post-selection coin. The `p=1` endpoint is the deterministic post-selected control and therefore has no site-observer measurement-frustration payload.


In [ ]:
NX = 20
NY = 40
NSHELL = 1
INIT_MODE = 'default'
REQUESTED_SAMPLES = 10
POSTSELECT_PROBABILITIES = (0.0, 0.1, 0.25, 0.5, 0.75, 0.9, 0.95, 0.99, 1.0)
ALPHA_1 = 1.0
ALPHA_2 = 30.0
DW_TRUNCATION = True
DTYPE = 'complex128'
DEVICE = 'cuda:0'
BACKEND = 'local'
SEQUENCE = 'raster_y'
TRIAL_ORBITALS = 'X'
N_A = 0.5
AUTOTUNE_ENTROPY = True
ENTROPY_AUTOTUNE_REPEAT = 1
SAMPLE_CHUNK_CANDIDATES = (1, 2, 4, 8, 16)
Y0_CHUNK_CANDIDATES = (1, 2, 4, 6, 8, 10, 12, 16, 20, 24, 32, 'ny')
MEMORY_SAFETY_FRACTION = 0.85
EIGH_MEMORY_MULTIPLIER = 8.0
OVERWRITE = True
SKIP_COMPLETED = True
CONFIG_LIMIT = None  # Set to 2 for a p=0,0.1 smoke run before the full sweep.
PROTOCOL_FAMILY = 'partial_postselection_sweep'


def format_probability_tag(p: float) -> str:
    return f'{float(p):g}'


def make_protocol_config(p: float) -> dict[str, Any]:
    p = float(p)
    ptag = format_probability_tag(p)
    if p == 1.0:
        return {
            'protocol': 'postselect',
            'protocol_display': 'postselect_psp1',
            'run_label': 'postselect_psp1',
            'postselect': True,
            'perfect_correction': False,
            'postselect_probability': 1.0,
            'samples': REQUESTED_SAMPLES,
            'batch_size': 1,
        }
    return {
        'protocol': 'perfect_correction',
        'protocol_display': f'partial_psp{ptag}',
        'run_label': f'partial_psp{ptag}',
        'postselect': False,
        'perfect_correction': True,
        'postselect_probability': p,
        'samples': REQUESTED_SAMPLES,
        'batch_size': None,
    }

ALL_CONFIGS = [
    {
        'Nx': NX,
        'Ny': NY,
        'cycles': NY,
        'nshell': NSHELL,
        'init_mode': INIT_MODE,
        'protocol_family': PROTOCOL_FAMILY,
        **make_protocol_config(p),
    }
    for p in POSTSELECT_PROBABILITIES
]
CONFIGS = ALL_CONFIGS if CONFIG_LIMIT is None else ALL_CONFIGS[: int(CONFIG_LIMIT)]

GPU_DATA_ROOT = BUNDLE_ROOT / 'gpu_data'
CAMPAIGN_NAME = 'streaming_covariance_observables'
OUTPUT_ROOT = GPU_DATA_ROOT / CAMPAIGN_NAME
CAMPAIGN_ID = 'N20x40_nsh1_dwtrunc1_alpha2-30_init-default_S10_C40_partial-postselect-psweep'
CAMPAIGN_ROOT = OUTPUT_ROOT / 'campaigns' / CAMPAIGN_ID
RUNS_ROOT = CAMPAIGN_ROOT / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)

CANONICAL_ENTRY_POINT = 'classA_U1FGTN_gpu.run_markov_circuit'
BASE_SAVED_OBSERVABLES = [
    'frob_norm',
    'frob_successive_delta',
    'entropy_y0avg_vs_ay',
    'real_space_chern',
    'xavg_square_correlator_vs_ry',
    'local_charge_cell',
]
BORN_RULE_ONLY_OBSERVABLES = [
    'born_rule_probabilities',
]
PERFECT_CORRECTION_ONLY_OBSERVABLES = [
    's_Ap',
    's_Am',
    's_Bp',
    's_Bm',
    'frustration_min',
    'frustration_avg',
    'frustration_prod',
    'postselect_forced_site',
]
SAVED_OBSERVABLES_BY_PROTOCOL = {
    'partial_postselection_p_lt_1': [*BASE_SAVED_OBSERVABLES, *BORN_RULE_ONLY_OBSERVABLES, *PERFECT_CORRECTION_ONLY_OBSERVABLES],
    'postselect_p_eq_1': [*BASE_SAVED_OBSERVABLES, *BORN_RULE_ONLY_OBSERVABLES],
}


def saved_observables_for_config(
    cfg: dict[str, Any],
    *,
    born_rule_probabilities_enabled: bool = True,
) -> list[str]:
    observables = list(BASE_SAVED_OBSERVABLES)
    if born_rule_probabilities_enabled:
        observables.extend(BORN_RULE_ONLY_OBSERVABLES)
    if bool(cfg['perfect_correction']):
        observables.extend(PERFECT_CORRECTION_ONLY_OBSERVABLES)
    return observables


def observable_collection_mode_for_config(cfg: dict[str, Any]) -> str:
    if bool(cfg['perfect_correction']):
        return 'cycle_observer_and_site_observer_streaming_with_postselect_forced_site_mask'
    return 'cycle_observer_streaming_postselected_endpoint'


def estimate_run_storage_bytes(cfg: dict[str, Any]) -> int:
    nx = int(cfg['Nx'])
    ny = int(cfg['Ny'])
    cycles = int(cfg['cycles'])
    samples_actual = expected_samples_for_config(cfg)
    scalars = samples_actual * cycles * 3
    curves = samples_actual * cycles * 2 * (ny // 2 + 1)
    field_count = 5 + (8 if bool(cfg['perfect_correction']) else 0)
    fields = samples_actual * cycles * field_count * nx * ny
    return int(8 * (scalars + curves + fields))

config_rows = []
for cfg in CONFIGS:
    config_rows.append({
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'cycles': int(cfg['cycles']),
        'nshell': int(cfg['nshell']),
        'protocol_family': cfg['protocol_family'],
        'protocol_display': cfg['protocol_display'],
        'run_label': cfg['run_label'],
        'postselect_probability': float(cfg['postselect_probability']),
        'postselect': bool(cfg['postselect']),
        'perfect_correction': bool(cfg['perfect_correction']),
        'requested_samples': int(cfg['samples']),
        'actual_samples': expected_samples_for_config(cfg),
        'batch_size': cfg['batch_size'],
        'estimated_storage_gib_float64': estimate_run_storage_bytes(cfg) / 1024**3,
    })

config_df = pd.DataFrame(config_rows)
display(Markdown('### Campaign config'))
display(config_df)
print(json.dumps({
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'postselect_probabilities': list(POSTSELECT_PROBABILITIES),
    'configs': CONFIGS,
    'covariance_history_saved': False,
    'estimated_storage_gib_float64': float(config_df['estimated_storage_gib_float64'].sum()),
    'protocol_note': 'p<1 uses perfect_correction with a per-site postselect_probability coin; p=1 is deterministic post-selection and actual_samples=1.',
}, indent=2))


In [ ]:

def p_tag_for_config(cfg: dict[str, Any]) -> str:
    return format_probability_tag(float(cfg['postselect_probability']))


def sweep_config_id(cfg: dict[str, Any]) -> str:
    return f"N{int(cfg['Nx'])}x{int(cfg['Ny'])}_{str(cfg['run_label'])}"


def run_output_dir(runs_root: Path, cfg: dict[str, Any]) -> Path:
    return Path(runs_root) / (
        f"N{int(cfg['Nx'])}x{int(cfg['Ny'])}_nsh{int(cfg['nshell'])}_"
        f"init-{cfg['init_mode']}_{str(cfg['run_label'])}"
    )

CHECKPOINT_SCHEMA_VERSION = 1


def update_gpu_data_index() -> None:
    index_path = GPU_DATA_ROOT / 'index.json'
    if index_path.exists():
        payload = json.loads(index_path.read_text())
    else:
        payload = {'bundle_root': str(BUNDLE_ROOT), 'gpu_data_root': str(GPU_DATA_ROOT), 'campaigns': []}
    campaigns = payload.setdefault('campaigns', [])
    campaigns[:] = [item for item in campaigns if item.get('name') != CAMPAIGN_NAME]
    campaigns.append({
        'name': CAMPAIGN_NAME,
        'kind': 'streamed_covariance_observables',
        'manifest': f'{CAMPAIGN_NAME}/latest_campaign.json',
    })
    payload['bundle_root'] = str(BUNDLE_ROOT)
    payload['gpu_data_root'] = str(GPU_DATA_ROOT)
    write_json_atomic(index_path, payload)


def assert_overwrite_allowed(path: Path) -> None:
    if path.exists() and not OVERWRITE:
        raise FileExistsError(f'Path already exists and OVERWRITE=False: {path}')


def expected_output_paths(
    cfg: dict[str, Any],
    out_dir: Path,
    *,
    born_rule_probabilities_enabled: bool = True,
) -> dict[str, Path]:
    paths = {
        'metrics_path': out_dir / 'scalar_metrics.csv',
        'entropy_path': out_dir / 'entropy_y0avg_vs_ay.npz',
        'corr_path': out_dir / 'xavg_square_correlator_vs_ry.npz',
        'charge_path': out_dir / 'local_charge_cell.npz',
        'summary_path': out_dir / 'run_summary.json',
        'latest_path': out_dir / 'latest_run.json',
    }
    if born_rule_probabilities_enabled:
        paths['born_probabilities_path'] = out_dir / 'born_rule_probabilities.npz'
    if bool(cfg['perfect_correction']):
        paths['frustration_path'] = out_dir / 'measurement_frustration.npz'
    return paths


def expected_checkpoint_paths(out_dir: Path) -> dict[str, Path]:
    return {
        'manifest_path': out_dir / 'observable_checkpoint_manifest.json',
        'checkpoint_dir': out_dir / 'observable_checkpoint_batches',
    }


def timestamp_utc() -> str:
    return time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime())


def file_size_bytes(path: Path) -> int:
    return int(path.stat().st_size)


def resolved_batch_plan_for_config(
    cfg: dict[str, Any],
    *,
    model: classA_U1FGTN_gpu,
    expected_samples: int,
    cycles: int,
) -> dict[str, Any]:
    if bool(cfg['postselect']):
        return {
            'batch_size': 1,
            'batch_size_mode': 'postselect',
            'batch_size_auto_info': None,
        }
    if cfg['batch_size'] is None:
        batch_size, batch_size_auto_info = model._auto_batch_size_for_a100_40gb(
            samples=expected_samples,
            cycles=cycles,
            store_mode='final',
            snapshot_cycles=None,
            init_mode=str(cfg['init_mode']),
        )
        return {
            'batch_size': int(batch_size),
            'batch_size_mode': 'auto_a100_40gb',
            'batch_size_auto_info': batch_size_auto_info,
        }
    return {
        'batch_size': int(cfg['batch_size']),
        'batch_size_mode': 'explicit',
        'batch_size_auto_info': None,
    }


def checkpoint_metadata_for_config(
    cfg: dict[str, Any],
    *,
    expected_samples: int,
    cycles: int,
    dw_meta: dict[str, Any],
    batch_plan: dict[str, Any],
) -> dict[str, Any]:
    return {
        'schema_version': CHECKPOINT_SCHEMA_VERSION,
        'helper_version': HELPER_VERSION,
        'helper_module_path': str(HELPER_MODULE_PATH),
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'cycles': int(cycles),
        'nshell': int(cfg['nshell']),
        'init_mode': str(cfg['init_mode']),
        'protocol': str(cfg['protocol']),
        'protocol_family': str(cfg['protocol_family']),
        'protocol_display': str(cfg['protocol_display']),
        'run_label': str(cfg['run_label']),
        'postselect_probability': float(cfg['postselect_probability']),
        'postselect': bool(cfg['postselect']),
        'perfect_correction': bool(cfg['perfect_correction']),
        'samples_requested': int(cfg['samples']),
        'samples_expected': int(expected_samples),
        'alpha_1': float(ALPHA_1),
        'alpha_2': float(ALPHA_2),
        'dw_truncation': bool(DW_TRUNCATION),
        'dtype': DTYPE,
        'device': DEVICE,
        'backend': BACKEND,
        'sequence': SEQUENCE,
        'n_a': float(N_A),
        'trial_orbitals': TRIAL_ORBITALS,
        'batch_size_requested': cfg['batch_size'],
        'resolved_batch_size': int(batch_plan['batch_size']),
        'batch_size_mode': str(batch_plan['batch_size_mode']),
        'dw_loc': dw_meta['dw_loc'],
        'topological_x_range': dw_meta['topological_x_range'],
        'trivial_x_segments': dw_meta['trivial_x_segments'],
    }


def make_checkpoint_manifest(
    *,
    checkpoint_metadata: dict[str, Any],
    batch_plan: dict[str, Any],
    observer: StreamingCovarianceObservables | None,
    status: str,
    completed_batches: list[int] | None = None,
    batch_records: dict[str, Any] | None = None,
    error: dict[str, Any] | None = None,
) -> dict[str, Any]:
    now = timestamp_utc()
    return {
        'schema_version': CHECKPOINT_SCHEMA_VERSION,
        'status': str(status),
        'created_at': now,
        'updated_at': now,
        'checkpoint_metadata': checkpoint_metadata,
        'batch_plan': batch_plan,
        'completed_batches': sorted(int(idx) for idx in (completed_batches or [])),
        'batch_records': batch_records or {},
        'observer_state': None if observer is None else observer.checkpoint_state_payload(),
        'error': error,
    }


def load_checkpoint_manifest(paths: dict[str, Path]) -> dict[str, Any] | None:
    manifest_path = paths['manifest_path']
    if not manifest_path.exists():
        return None
    try:
        return json.loads(manifest_path.read_text())
    except Exception as exc:
        print(f'[resume] ignoring unreadable checkpoint manifest {manifest_path}: {exc}')
        return {'_manifest_error': str(exc)}


def write_checkpoint_manifest(paths: dict[str, Path], manifest: dict[str, Any]) -> None:
    manifest['updated_at'] = timestamp_utc()
    write_json_atomic(paths['manifest_path'], manifest)


def reset_checkpoint_state(paths: dict[str, Path]) -> None:
    manifest_path = paths['manifest_path']
    checkpoint_dir = paths['checkpoint_dir']
    if manifest_path.exists():
        manifest_path.unlink()
    if checkpoint_dir.exists():
        shutil.rmtree(checkpoint_dir)


def remove_checkpoint_batches(paths: dict[str, Path]) -> None:
    checkpoint_dir = paths['checkpoint_dir']
    if checkpoint_dir.exists():
        shutil.rmtree(checkpoint_dir)




def summary_born_rule_probabilities_enabled(summary: dict[str, Any]) -> bool:
    if 'born_rule_probabilities_enabled' in summary:
        return bool(summary['born_rule_probabilities_enabled'])
    born_relative = summary.get('born_rule_probabilities_relative')
    return born_relative not in (None, '')

def build_result_from_summary(cfg: dict[str, Any], out_dir: Path, summary: dict[str, Any]) -> dict[str, Any]:
    protocol = str(cfg['protocol'])
    return {
        'config_id': sweep_config_id(cfg),
        'geometry_key': geometry_key(cfg['Nx'], cfg['Ny']),
        'protocol': protocol,
        'protocol_family': cfg['protocol_family'],
        'protocol_display': cfg['protocol_display'],
        'run_label': cfg['run_label'],
        'postselect_probability': float(summary.get('postselect_probability', cfg['postselect_probability'])),
        'Nx': int(summary['Nx']),
        'Ny': int(summary['Ny']),
        'cycles': int(summary['cycles']),
        'nshell': int(summary['nshell']),
        'samples_requested': int(summary['samples_requested']),
        'samples_actual': int(summary['samples_actual']),
        'batch_size_requested': summary['batch_size_requested'],
        'batch_size': int(summary['batch_size']),
        'batch_size_mode': summary['batch_size_mode'],
        'run_dir_relative': summary['run_dir_relative'],
        'run_summary_relative': rel_to_root(out_dir / 'run_summary.json', GPU_DATA_ROOT),
        'saved_total_bytes': int(summary['saved_total_bytes']),
        'scalar_metrics_bytes': int(summary['saved_file_sizes_bytes']['scalar_metrics']),
        'entropy_bytes': int(summary['saved_file_sizes_bytes']['entropy_y0avg_vs_ay']),
        'square_corr_bytes': int(summary['saved_file_sizes_bytes']['xavg_square_correlator_vs_ry']),
        'local_charge_cell_bytes': int(summary['saved_file_sizes_bytes']['local_charge_cell']),
        'born_rule_probabilities_bytes': int(summary['saved_file_sizes_bytes'].get('born_rule_probabilities', 0)),
        'measurement_frustration_bytes': int(summary['saved_file_sizes_bytes'].get('measurement_frustration', 0)),
        'dw_loc': summary['dw_loc'],
        'topological_x_range': summary['topological_x_range'],
        'trivial_x_segments': summary['trivial_x_segments'],
    }


def try_load_completed_result(cfg: dict[str, Any], out_dir: Path) -> dict[str, Any] | None:
    summary_path = out_dir / 'run_summary.json'
    if not summary_path.exists():
        return None
    try:
        summary = json.loads(summary_path.read_text())
    except Exception as exc:
        print(f'[resume] ignoring unreadable completed summary {summary_path}: {exc}')
        return None
    paths = expected_output_paths(
        cfg,
        out_dir,
        born_rule_probabilities_enabled=summary_born_rule_probabilities_enabled(summary),
    )
    missing = [path for path in paths.values() if not path.exists()]
    if missing:
        return None
    protocol = str(cfg['protocol'])
    print(
        f"[skip] Nx={cfg['Nx']} Ny={cfg['Ny']} nshell={cfg['nshell']} init={cfg['init_mode']} "
        f"protocol_display={cfg['protocol_display']} p={cfg['postselect_probability']} output={out_dir}"
    )
    return build_result_from_summary(cfg, out_dir, summary)


def inspect_run_state(
    cfg: dict[str, Any],
    *,
    out_dir: Path,
    checkpoint_metadata: dict[str, Any],
) -> dict[str, Any]:
    del cfg
    checkpoint_paths = expected_checkpoint_paths(out_dir)
    manifest = load_checkpoint_manifest(checkpoint_paths)
    if manifest is None:
        return {
            'status': 'pending',
            'checkpoint_paths': checkpoint_paths,
            'manifest': None,
        }
    if '_manifest_error' in manifest:
        reset_checkpoint_state(checkpoint_paths)
        return {
            'status': 'pending',
            'checkpoint_paths': checkpoint_paths,
            'manifest': None,
        }
    if manifest.get('checkpoint_metadata') != checkpoint_metadata:
        print('[resume] checkpoint metadata changed; discarding stale checkpoint state and starting over')
        reset_checkpoint_state(checkpoint_paths)
        return {
            'status': 'pending',
            'checkpoint_paths': checkpoint_paths,
            'manifest': None,
        }

    batch_records = manifest.get('batch_records', {})
    usable_batches = []
    usable_records = {}
    for idx in sorted(int(x) for x in manifest.get('completed_batches', [])):
        record = batch_records.get(str(idx))
        if not isinstance(record, dict):
            continue
        filename = record.get('filename')
        if not filename:
            continue
        shard_path = checkpoint_paths['checkpoint_dir'] / str(filename)
        if shard_path.exists():
            usable_batches.append(idx)
            usable_records[str(idx)] = record
    if not usable_batches:
        print('[resume] no usable completed checkpoint batches found; restarting config from scratch')
        reset_checkpoint_state(checkpoint_paths)
        return {
            'status': 'pending',
            'checkpoint_paths': checkpoint_paths,
            'manifest': None,
        }

    manifest['completed_batches'] = usable_batches
    manifest['batch_records'] = usable_records
    manifest['status'] = 'partial'
    write_checkpoint_manifest(checkpoint_paths, manifest)
    return {
        'status': 'partial',
        'checkpoint_paths': checkpoint_paths,
        'manifest': manifest,
    }


def run_config_payload(
    cfg: dict[str, Any],
    *,
    actual_samples: int,
    dw_meta: dict[str, Any],
    born_rule_probabilities_enabled: bool,
) -> dict[str, Any]:
    return {
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'cycles': int(cfg['cycles']),
        'nshell': int(cfg['nshell']),
        'init_mode': str(cfg['init_mode']),
        'protocol': str(cfg['protocol']),
        'protocol_family': str(cfg['protocol_family']),
        'protocol_display': str(cfg['protocol_display']),
        'run_label': str(cfg['run_label']),
        'postselect_probability': float(cfg['postselect_probability']),
        'postselect': bool(cfg['postselect']),
        'perfect_correction': bool(cfg['perfect_correction']),
        'samples_requested': int(cfg['samples']),
        'samples_actual': int(actual_samples),
        'alpha_1': float(ALPHA_1),
        'alpha_2': float(ALPHA_2),
        'dw_truncation': bool(DW_TRUNCATION),
        'dtype': DTYPE,
        'device': DEVICE,
        'backend': BACKEND,
        'sequence': SEQUENCE,
        'batch_size_requested': cfg['batch_size'],
        'n_a': float(N_A),
        'trial_orbitals': TRIAL_ORBITALS,
        'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
        'covariance_history_saved': False,
        'observable_collection_mode': observable_collection_mode_for_config(cfg),
        'saved_observables': saved_observables_for_config(cfg, born_rule_probabilities_enabled=born_rule_probabilities_enabled),
        'dw_loc': dw_meta['dw_loc'],
        'topological_x_range': dw_meta['topological_x_range'],
        'trivial_x_segments': dw_meta['trivial_x_segments'],
    }


In [ ]:
def run_one_config(cfg: dict[str, Any]) -> dict[str, Any]:
    out_dir = run_output_dir(RUNS_ROOT, cfg)
    out_dir.mkdir(parents=True, exist_ok=True)
    if SKIP_COMPLETED:
        completed_result = try_load_completed_result(cfg, out_dir)
        if completed_result is not None:
            return completed_result
    expected_samples = expected_samples_for_config(cfg)
    protocol = str(cfg['protocol'])
    cycles = int(cfg['cycles'])
    print(
        f"[config] Nx={cfg['Nx']} Ny={cfg['Ny']} nshell={cfg['nshell']} init={cfg['init_mode']} "
        f"protocol_display={cfg['protocol_display']} p={cfg['postselect_probability']} requested_samples={cfg['samples']} actual_samples={expected_samples} "
        f"cycles={cycles} output={out_dir}"
    )

    model = classA_U1FGTN_gpu(
        Nx=int(cfg['Nx']),
        Ny=int(cfg['Ny']),
        DW=True,
        nshell=int(cfg['nshell']),
        filling_frac=0.5,
        alpha_1=ALPHA_1,
        alpha_2=ALPHA_2,
        trial_orbitals=TRIAL_ORBITALS,
        dw_truncation=DW_TRUNCATION,
        triv_region_local_mode=False,
        device=DEVICE,
        dtype=DTYPE,
        backend=BACKEND,
    )

    dw_meta = domain_wall_metadata(model)
    batch_plan = resolved_batch_plan_for_config(
        cfg,
        model=model,
        expected_samples=expected_samples,
        cycles=cycles,
    )
    checkpoint_metadata = checkpoint_metadata_for_config(
        cfg,
        expected_samples=expected_samples,
        cycles=cycles,
        dw_meta=dw_meta,
        batch_plan=batch_plan,
    )
    run_state = inspect_run_state(cfg, out_dir=out_dir, checkpoint_metadata=checkpoint_metadata)

    enable_born_probabilities = run_state['status'] != 'partial'
    if run_state['status'] == 'partial':
        print(
            f"[resume] resuming {sweep_config_id(cfg)} with "
            f"{len(run_state['manifest']['completed_batches'])} completed batches"
        )
        print('[resume] born-rule probabilities disabled for resumed config; they resume on the next fresh config')
    else:
        print(f"[resume] starting fresh for {sweep_config_id(cfg)}")

    observer = StreamingCovarianceObservables(
        nx=int(cfg['Nx']),
        ny=int(cfg['Ny']),
        cycles=cycles,
        samples_expected=expected_samples,
        protocol=protocol,
        nshell=int(cfg['nshell']),
        dw_loc=dw_meta['dw_loc'],
        autotune_entropy=AUTOTUNE_ENTROPY,
        sample_chunk_candidates=SAMPLE_CHUNK_CANDIDATES,
        y0_chunk_candidates=Y0_CHUNK_CANDIDATES,
        autotune_repeat=ENTROPY_AUTOTUNE_REPEAT,
        memory_safety_fraction=MEMORY_SAFETY_FRACTION,
        eigh_memory_multiplier=EIGH_MEMORY_MULTIPLIER,
        compute_local_chern_marker=False,
        enable_born_probabilities=enable_born_probabilities,
    )

    checkpoint_paths = run_state['checkpoint_paths']
    manifest = run_state['manifest']
    restored_batches = []
    if manifest is not None:
        observer.restore_checkpoint_state(manifest.get('observer_state'))
        for batch_index in sorted(int(x) for x in manifest.get('completed_batches', [])):
            record = manifest['batch_records'][str(batch_index)]
            observer.load_batch_checkpoint(
                checkpoint_paths['checkpoint_dir'] / str(record['filename']),
                checkpoint_metadata=checkpoint_metadata,
            )
            restored_batches.append(batch_index)
    else:
        manifest = make_checkpoint_manifest(
            checkpoint_metadata=checkpoint_metadata,
            batch_plan=batch_plan,
            observer=observer,
            status='running',
        )
        write_checkpoint_manifest(checkpoint_paths, manifest)

    manifest['status'] = 'running'
    manifest['error'] = None
    manifest['batch_plan'] = batch_plan
    manifest['checkpoint_metadata'] = checkpoint_metadata
    manifest['observer_state'] = observer.checkpoint_state_payload()
    write_checkpoint_manifest(checkpoint_paths, manifest)

    completed_batch_indices = set(int(x) for x in manifest.get('completed_batches', []))
    batch_records = dict(manifest.get('batch_records', {}))
    completed_progress = sum(
        int(batch_records[str(idx)]['batch_count']) * cycles
        for idx in completed_batch_indices
        if str(idx) in batch_records
    )

    observer_pbar = tqdm(
        total=expected_samples * cycles,
        desc=f"observer {sweep_config_id(cfg)}",
        unit='sample-cycle',
        leave=False,
    )
    if completed_progress:
        observer_pbar.update(completed_progress)

    base_cycle_observer = observer.make_cycle_observer(model, progress_bar=observer_pbar)

    def cycle_observer(*, cycle, G, batch_index, batch_start, batch_count):
        base_cycle_observer(
            cycle=cycle,
            G=G,
            batch_index=batch_index,
            batch_start=batch_start,
            batch_count=batch_count,
        )
        if int(cycle) != cycles or int(batch_index) in completed_batch_indices:
            return
        record = observer.export_batch_checkpoint(
            checkpoint_dir=checkpoint_paths['checkpoint_dir'],
            batch_index=int(batch_index),
            batch_start=int(batch_start),
            batch_count=int(batch_count),
            checkpoint_metadata=checkpoint_metadata,
        )
        completed_batch_indices.add(int(batch_index))
        batch_records[str(int(batch_index))] = {
            **record,
            'path_relative': rel_to_root(Path(record['path']), GPU_DATA_ROOT),
        }
        manifest['status'] = 'partial'
        manifest['completed_batches'] = sorted(completed_batch_indices)
        manifest['batch_records'] = batch_records
        manifest['observer_state'] = observer.checkpoint_state_payload()
        manifest['error'] = None
        write_checkpoint_manifest(checkpoint_paths, manifest)

    site_observer = observer.make_site_observer() if bool(cfg['perfect_correction']) else None
    try:
        result = model.run_markov_circuit(
            G_history=False,
            progress=False,
            cycles=cycles,
            postselect=bool(cfg['postselect']),
            postselect_probability=float(cfg['postselect_probability']),
            perfect_correction=bool(cfg['perfect_correction']),
            samples=int(cfg['samples']),
            init_mode=str(cfg['init_mode']),
            save=False,
            save_init=False,
            save_history_stride=None,
            n_a=N_A,
            sequence=SEQUENCE,
            batch_size=int(batch_plan['batch_size']),
            return_data=False,
            cycle_observer=cycle_observer,
            site_observer=site_observer,
            skip_batch_indices=sorted(completed_batch_indices),
        )
    except Exception as exc:
        manifest['status'] = 'failed'
        manifest['completed_batches'] = sorted(completed_batch_indices)
        manifest['batch_records'] = batch_records
        manifest['observer_state'] = observer.checkpoint_state_payload()
        manifest['error'] = {
            'type': type(exc).__name__,
            'message': str(exc),
            'timestamp_utc': timestamp_utc(),
        }
        write_checkpoint_manifest(checkpoint_paths, manifest)
        raise
    finally:
        observer_pbar.close()

    actual_samples = int(result['samples'])
    if actual_samples != expected_samples:
        raise RuntimeError(f'Expected {expected_samples} samples but run returned {actual_samples}.')
    observer.finalize(actual_samples=actual_samples)

    metrics_df = observer.metrics_dataframe().sort_values(['sample_index', 'cycle_label']).reset_index(drop=True)
    metrics_df['config_id'] = sweep_config_id(cfg)
    metrics_df['protocol_family'] = str(cfg['protocol_family'])
    metrics_df['protocol_display'] = str(cfg['protocol_display'])
    metrics_df['run_label'] = str(cfg['run_label'])
    metrics_df['postselect_probability'] = float(cfg['postselect_probability'])
    metrics_df['requested_samples'] = int(cfg['samples'])
    metrics_df['actual_samples'] = int(actual_samples)
    run_payload = run_config_payload(
        cfg,
        actual_samples=actual_samples,
        dw_meta=dw_meta,
        born_rule_probabilities_enabled=enable_born_probabilities,
    )

    output_paths = expected_output_paths(
        cfg,
        out_dir,
        born_rule_probabilities_enabled=enable_born_probabilities,
    )
    metrics_path = output_paths['metrics_path']
    entropy_path = output_paths['entropy_path']
    corr_path = output_paths['corr_path']
    charge_path = output_paths['charge_path']
    born_probabilities_path = output_paths.get('born_probabilities_path')
    frustration_path = output_paths.get('frustration_path')
    summary_path = output_paths['summary_path']
    latest_path = output_paths['latest_path']
    paths_to_overwrite = [metrics_path, entropy_path, corr_path, charge_path, summary_path, latest_path]
    if born_probabilities_path is not None:
        paths_to_overwrite.append(born_probabilities_path)
    if frustration_path is not None:
        paths_to_overwrite.append(frustration_path)
    for path in paths_to_overwrite:
        assert_overwrite_allowed(path)

    save_dataframe_atomic_csv(metrics_df, metrics_path)
    save_npz_atomic(entropy_path, **observer.entropy_npz_payload(config=run_payload))
    save_npz_atomic(corr_path, **observer.square_correlator_npz_payload(config=run_payload))
    save_npz_atomic(charge_path, **observer.local_charge_cell_npz_payload(config=run_payload))
    if born_probabilities_path is not None:
        save_npz_atomic(born_probabilities_path, **observer.born_rule_probabilities_npz_payload(config=run_payload))
    if frustration_path is not None:
        save_npz_atomic(frustration_path, **observer.measurement_frustration_npz_payload(config=run_payload))

    run_summary = {
        **run_payload,
        'run_dir': str(out_dir),
        'run_dir_relative': rel_to_root(out_dir, GPU_DATA_ROOT),
        'scalar_metrics_relative': rel_to_root(metrics_path, GPU_DATA_ROOT),
        'entropy_y0avg_vs_ay_relative': rel_to_root(entropy_path, GPU_DATA_ROOT),
        'xavg_square_correlator_vs_ry_relative': rel_to_root(corr_path, GPU_DATA_ROOT),
        'local_charge_cell_relative': rel_to_root(charge_path, GPU_DATA_ROOT),
        'born_rule_probabilities_enabled': bool(enable_born_probabilities),
        'born_rule_probabilities_relative': rel_to_root(born_probabilities_path, GPU_DATA_ROOT),
        'measurement_frustration_relative': rel_to_root(frustration_path, GPU_DATA_ROOT),
        'batch_size': int(result['batch_size']),
        'batch_size_mode': result['batch_size_mode'],
        'batch_size_auto_info': result.get('batch_size_auto_info'),
        'metric_row_count': int(len(metrics_df)),
        'helper_version': HELPER_VERSION,
        'helper_module_path': str(HELPER_MODULE_PATH),
        'checkpoint_manifest_relative': rel_to_root(checkpoint_paths['manifest_path'], GPU_DATA_ROOT),
        'resumed_completed_batches': restored_batches,
        'saved_file_sizes_bytes': {
            'scalar_metrics': file_size_bytes(metrics_path),
            'entropy_y0avg_vs_ay': file_size_bytes(entropy_path),
            'xavg_square_correlator_vs_ry': file_size_bytes(corr_path),
            'local_charge_cell': file_size_bytes(charge_path),
            **({'born_rule_probabilities': file_size_bytes(born_probabilities_path)} if born_probabilities_path is not None else {}),
            **({'measurement_frustration': file_size_bytes(frustration_path)} if frustration_path is not None else {}),
        },
        **observer.run_summary_metrics(),
    }
    run_summary['saved_total_bytes'] = int(sum(run_summary['saved_file_sizes_bytes'].values()))
    write_json_atomic(summary_path, run_summary)
    write_json_atomic(
        latest_path,
        {
            'run_summary_path': str(summary_path),
            'run_summary_path_relative': rel_to_root(summary_path, GPU_DATA_ROOT),
            **run_summary,
        },
    )

    manifest['status'] = 'completed'
    manifest['completed_batches'] = sorted(completed_batch_indices)
    manifest['batch_records'] = batch_records
    manifest['observer_state'] = observer.checkpoint_state_payload()
    manifest['error'] = None
    manifest['completed_at'] = timestamp_utc()
    manifest['final_outputs'] = {
        'summary_path_relative': rel_to_root(summary_path, GPU_DATA_ROOT),
        'latest_path_relative': rel_to_root(latest_path, GPU_DATA_ROOT),
    }
    write_checkpoint_manifest(checkpoint_paths, manifest)
    remove_checkpoint_batches(checkpoint_paths)

    return {
        'config_id': sweep_config_id(cfg),
        'geometry_key': geometry_key(cfg['Nx'], cfg['Ny']),
        'protocol': protocol,
        'protocol_family': cfg['protocol_family'],
        'protocol_display': cfg['protocol_display'],
        'run_label': cfg['run_label'],
        'postselect_probability': float(cfg['postselect_probability']),
        'Nx': int(cfg['Nx']),
        'Ny': int(cfg['Ny']),
        'cycles': cycles,
        'nshell': int(cfg['nshell']),
        'samples_requested': int(cfg['samples']),
        'samples_actual': actual_samples,
        'batch_size_requested': cfg['batch_size'],
        'batch_size': int(result['batch_size']),
        'batch_size_mode': result['batch_size_mode'],
        'run_dir_relative': rel_to_root(out_dir, GPU_DATA_ROOT),
        'run_summary_relative': rel_to_root(summary_path, GPU_DATA_ROOT),
        'saved_total_bytes': int(run_summary['saved_total_bytes']),
        'scalar_metrics_bytes': int(run_summary['saved_file_sizes_bytes']['scalar_metrics']),
        'entropy_bytes': int(run_summary['saved_file_sizes_bytes']['entropy_y0avg_vs_ay']),
        'square_corr_bytes': int(run_summary['saved_file_sizes_bytes']['xavg_square_correlator_vs_ry']),
        'local_charge_cell_bytes': int(run_summary['saved_file_sizes_bytes']['local_charge_cell']),
        'born_rule_probabilities_bytes': int(run_summary['saved_file_sizes_bytes'].get('born_rule_probabilities', 0)),
        'measurement_frustration_bytes': int(run_summary['saved_file_sizes_bytes'].get('measurement_frustration', 0)),
        'dw_loc': dw_meta['dw_loc'],
        'topological_x_range': dw_meta['topological_x_range'],
        'trivial_x_segments': dw_meta['trivial_x_segments'],
    }


In [ ]:
all_results = []
campaign_start = time.perf_counter()
for cfg in CONFIGS:
    all_results.append(run_one_config(cfg))
elapsed_s = float(time.perf_counter() - campaign_start)

run_index = pd.DataFrame(all_results).sort_values(['postselect_probability']).reset_index(drop=True)
run_index_path = CAMPAIGN_ROOT / 'run_index.csv'
assert_overwrite_allowed(run_index_path)
save_dataframe_atomic_csv(run_index, run_index_path)

campaign_manifest = {
    'campaign_name': CAMPAIGN_NAME,
    'campaign_id': CAMPAIGN_ID,
    'protocol_family': PROTOCOL_FAMILY,
    'canonical_dynamics_entry_point': CANONICAL_ENTRY_POINT,
    'helper_version': HELPER_VERSION,
    'bundle_root': str(BUNDLE_ROOT),
    'project_root': str(PROJECT_ROOT),
    'gpu_data_root': str(GPU_DATA_ROOT),
    'output_root': str(OUTPUT_ROOT),
    'campaign_root': str(CAMPAIGN_ROOT),
    'runs_root': str(RUNS_ROOT),
    'output_root_relative': rel_to_root(OUTPUT_ROOT, GPU_DATA_ROOT),
    'campaign_root_relative': rel_to_root(CAMPAIGN_ROOT, GPU_DATA_ROOT),
    'runs_root_relative': rel_to_root(RUNS_ROOT, GPU_DATA_ROOT),
    'run_index_relative': rel_to_root(run_index_path, GPU_DATA_ROOT),
    'configs': CONFIGS,
    'results': all_results,
    'postselect_probabilities': list(POSTSELECT_PROBABILITIES),
    'requested_samples': REQUESTED_SAMPLES,
    'skip_completed': SKIP_COMPLETED,
    'dtype': DTYPE,
    'init_mode': INIT_MODE,
    'alpha_1': float(ALPHA_1),
    'alpha_2': float(ALPHA_2),
    'dw_truncation': DW_TRUNCATION,
    'trial_orbitals': TRIAL_ORBITALS,
    'backend': BACKEND,
    'sequence': SEQUENCE,
    'covariance_history_saved': False,
    'observable_collection_mode': 'partial_postselection_p_sweep_streaming_observers_without_local_chern_marker',
    'saved_observables_by_protocol': SAVED_OBSERVABLES_BY_PROTOCOL,
    'campaign_outputs': {
        'run_index_relative': rel_to_root(run_index_path, GPU_DATA_ROOT),
    },
    'saved_total_bytes': int(run_index['saved_total_bytes'].sum()),
    'elapsed_seconds': elapsed_s,
    'note': 'p<1 uses trajectory-resolved perfect correction with per-site postselect_probability; p=1 is deterministic post-selection and actual_samples=1.',
}
campaign_manifest_path = CAMPAIGN_ROOT / 'campaign_manifest.json'
assert_overwrite_allowed(campaign_manifest_path)
write_json_atomic(campaign_manifest_path, campaign_manifest)

latest_campaign_path = OUTPUT_ROOT / 'latest_campaign.json'
assert_overwrite_allowed(latest_campaign_path)
write_json_atomic(
    latest_campaign_path,
    {
        'campaign_name': CAMPAIGN_NAME,
        'campaign_id': CAMPAIGN_ID,
        'protocol_family': PROTOCOL_FAMILY,
        'manifest': rel_to_root(campaign_manifest_path, GPU_DATA_ROOT),
        'campaign_manifest_path': str(campaign_manifest_path),
        'campaign_manifest_path_relative': rel_to_root(campaign_manifest_path, GPU_DATA_ROOT),
    },
)
update_gpu_data_index()

print(f'[done] campaign manifest: {campaign_manifest_path}')
print(f'[done] latest campaign pointer: {latest_campaign_path}')
print(f'[done] run index: {run_index_path}')
display(Markdown('### Saved run index'))
display(run_index)


In [ ]:
from IPython.display import Javascript, display

try:
    display(Javascript('google.colab.kernel.disconnect()'))
except Exception as exc:
    print(f'[disconnect] skipped: {exc}')
